# ZelusBench: Selective Attention — NoisyPins num_points=15 while randomizing everything else.More points = more irrelevant information to filter.- 15 scenarios

In [ ]:
import kaggle_benchmarks as kbenchimport numpy as npimport randomimport timefrom zelusbench.scenarios.config import ScenarioConfig, QueryTypefrom zelusbench.scenarios.generator import ScenarioGeneratorfrom zelusbench.evaluation.scorer import score_query, score_responseNUM_POINTS = 15SEEDS = 15print(f"ZelusBench Selective Attention — Noisy (num_points={NUM_POINTS})")

In [ ]:
@kbench.task(name="zelusbench_attn_selective_noisy")def zelusbench_attn_selective_noisy(llm) -> tuple[float, float]:    _start = time.time()    print(f"Running {SEEDS} scenarios...")    print("=" * 60)    all_scores = []    for i in range(SEEDS):        bg_rng = random.Random(i * 7919)        cfg = ScenarioConfig.randomize_except(bg_rng, pinned={            "num_points": NUM_POINTS,            "num_queries": 3,            "seed": i,        })        scenario = ScenarioGenerator(cfg).generate(f"selective_{i}")        response = llm.prompt(scenario.prompt)        scores = score_response(response, scenario)        all_scores.extend(scores)        avg = float(np.mean([s.score for s in scores]))        bg = f"dim={cfg.dim} lb={cfg.leaf_bias} tp={cfg.transform_prob}"        print(f"  [{i+1}/{SEEDS}] avg={avg:.2f} | {bg}")    overall = float(np.mean([s.score for s in all_scores]))    std = float(np.std([s.score for s in all_scores]))    print(f"\nOverall: {overall:.3f} +/- {std:.3f} | {len(all_scores)} queries | {time.time()-_start:.1f}s")    kbench.assertions.assert_true(overall >= 0, expectation=f"Valid answers (got {overall:.3f})")    return overall, stdzelusbench_attn_selective_noisy.run(llm=kbench.llm)

In [ ]:
%choose zelusbench_attn_selective_noisy